# Datamine MINEWD Process Example

This notebook demonstrates how to configure and run the **`minewd`** process wrapper in `dmstudio`.

## Process Description

## MINEWD

Creates LG 'Additional Arcs' for mining width.

The input file is a block model file containing the ore blocks of the main reef. In addition, it should have an **OREVAL** field which contains a value which is to be compared with the **OREMIN** parameter. The **MINWID** parameter defines the required minimum mining width; the WIDDIR parameter defines the bearing in degrees, clockwise from the positive Y-axis (North) along which the minimum mining width extension is to be made. For example, if the main reef lies North-West to South-East and dips towards the North-East, this parameter would be set to 45 degrees. The **OREMIN** parameter defines the minimum value of the field **OREVAL** for a cell to be treated as ore.

The input block model must be a regular model with no sub-cells.

This process generates a file of Lerchs-Grossmann structure arcs for input to LGST as 'Additional arcs' for the structure of a block model. These arcs have the effect of ensuring that a required minimum mining width is honoured at the bottom of the pit during the Lerchs-Grossmann optimization. This approach is only appropriate when the main reef meets all of these criteria:

* Is thin compared with the minimum mining width.

* Dips at angle which is steeper than the required mining slope.

* Has a clearly defined strike d

By 'main reef' we mean a reef (ore body) which is mined at the full depth of the mine.

Other reefs may be mined as 'bonuses', and will be considered in the Lerchs-Grossmann optimization. It is necessary to decide on which side of the main reef the extra waste is to be removed to provide the necessary mining width.

Given this decision, **MINEWD** will set up structure arcs which cause strings of blocks of the length required to reach the minimum mining width to be treated by the optimization module as though they were one block. That is, either all blocks in a string are mined or none of them are mined. This ensures that the bottom of the pit is at least the minimum mining width across.

### Input Files:

* **in** (Undefined):
  Input model file containing the ore blocks of the main reef. This must be a regular
  model with no sub-cells.
  Required=Yes

### Output Files:

* **out** (Undefined):
  Output additional arcs file for input to LGST.
  Required=Yes

### Fields:

* **oreval** (Undefined : Undefined):
  A field in the model file which contains a value which is to be compared with

## **OREMIN**.

  Default=Undefined
  Required=No

### Parameters:

* **minwid**:
  Minimum mining width.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **widdir**:
  Bearing in degrees clockwise from the postive Y-direction (North) along which the
  minimum mining width extension is to be made. e.g. If the main reef lies North-West to
  South-East and dips down towards the North-East, this might be 45 degrees.
  Range=Undefined
  Values=Undefined
  Default=North
  Required=Yes

* **oremin**:
  Minimum value of field **OREVAL** for a cell to be treated as ore.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('minewd')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute minewd
print("Running minewd...")
dm_cmd.minewd(
    in_i='t_assays',  # required
    out_o='t_minewd_out',  # required
    oreval_f='optional',  # required
    minwid_p='required_val',  # required
    widdir_p='required_val',  # required
    # oremin_p='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("minewd execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_minewd_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")